In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
greenhouse_bronze = spark.table(
    "job_market.bronze.greenhouse_jobs"
)

print("Greenhouse Bronze records:", greenhouse_bronze.count())

greenhouse_bronze.printSchema()

Greenhouse Bronze records: 1583
root
 |-- source: string (nullable = true)
 |-- company: string (nullable = true)
 |-- board_token: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- job_id: long (nullable = true)
 |-- job_title: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |-- departments: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- child_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- parent_id: long (nullable = true)
 |-- offices: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- child_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- locati

In [0]:
display(
    greenhouse_bronze.select(
        "company",
        "job_id",
        "job_title",
        "location",
        "departments",
        "offices",
        "employment",
        "education"
    ).limit(10)
)

company,job_id,job_title,location,departments,offices,employment,education
Cloudflare,7695702,"Account Executive, FedCiv",List(Hybrid),"List(List(List(), 70660, Field Sales, 29066))","List(List(List(), 19994, Washington, DC, United States, Washington, DC, 48752))",null,null
Cloudflare,8172582,"Account Executive, Indonesia",List(Hybrid),"List(List(List(), 70658, Mid Market, 29066))","List(List(List(), 19992, Singapore, Singapore, 74844))",null,null
Cloudflare,8097321,AI Security Research & Red Team Engineer,List(Hybrid),"List(List(List(), 54562, Security, null))","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752), List(List(), 19990, New York, New York, United States, New York, NY, 48752))",null,null
Cloudflare,8144669,"Associate General Counsel, Privacy Compliance",List(Hybrid),"List(List(List(), 29069, Legal, Policy, Trust & Safety, null))","List(List(List(), 19988, London, United Kingdom, London, United Kingdom, 74843))",null,null
Cloudflare,7167659,Business Development Representative - Bahasa Indonesia Speaking,List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 19992, Singapore, Singapore, 74844))",null,null
Cloudflare,6557318,Business Development Representative (BDR),List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))",null,education_optional
Cloudflare,8234270,"Business Development Representative (BDR), LATAM Portuguese Speaking",List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))",null,education_optional
Cloudflare,7991345,"Business Development Representative (BDR), LATAM Spanish Speaking",List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))",null,education_optional
Cloudflare,8143314,"Business Development Representative, Bengaluru",List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 83149, null, Remote India, 74844))",null,null
Cloudflare,8143252,Business Development Representative (Cantonese Speaking),List(Hybrid),"List(List(List(), 70657, BDR, 29066))","List(List(List(), 48755, Beijing Shi, China, Beijing, China, 74844), List(List(), 19992, Singapore, Singapore, 74844))",null,null


In [0]:
greenhouse_silver = (
    greenhouse_bronze
    .withColumn(
        "job_title",
        F.trim(
            F.regexp_replace(
                F.col("job_title"),
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "location",
        F.trim(
            F.regexp_replace(
                F.col("location")["name"],
                r"\s+",
                " "
            )
        )
    )
    .withColumn(
        "content",
        F.trim(F.col("content"))
    )
)

In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "location_normalized",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("location"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "department",
        F.when(
            F.size("departments") > 0,
            F.element_at(
                F.transform(
                    "departments",
                    lambda x: x["name"]
                ),
                1
            )
        )
    )
)

In [0]:
display(
    greenhouse_silver.select(
        "company",
        "job_title",
        "departments",
        "department"
    ).limit(20)
)

company,job_title,departments,department
Cloudflare,"Account Executive, FedCiv","List(List(List(), 70660, Field Sales, 29066))",Field Sales
Cloudflare,"Account Executive, Indonesia","List(List(List(), 70658, Mid Market, 29066))",Mid Market
Cloudflare,AI Security Research & Red Team Engineer,"List(List(List(), 54562, Security, null))",Security
Cloudflare,"Associate General Counsel, Privacy Compliance","List(List(List(), 29069, Legal, Policy, Trust & Safety, null))","Legal, Policy, Trust & Safety"
Cloudflare,Business Development Representative - Bahasa Indonesia Speaking,"List(List(List(), 70657, BDR, 29066))",BDR
Cloudflare,Business Development Representative (BDR),"List(List(List(), 70657, BDR, 29066))",BDR
Cloudflare,"Business Development Representative (BDR), LATAM Portuguese Speaking","List(List(List(), 70657, BDR, 29066))",BDR
Cloudflare,"Business Development Representative (BDR), LATAM Spanish Speaking","List(List(List(), 70657, BDR, 29066))",BDR
Cloudflare,"Business Development Representative, Bengaluru","List(List(List(), 70657, BDR, 29066))",BDR
Cloudflare,Business Development Representative (Cantonese Speaking),"List(List(List(), 70657, BDR, 29066))",BDR


In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "office",
        F.when(
            F.size("offices") > 0,
            F.element_at(
                F.transform(
                    "offices",
                    lambda x: x["name"]
                ),
                1
            )
        )
    )
)

In [0]:
display(
    greenhouse_silver.select(
        "company",
        "job_title",
        "offices",
        "office"
    ).limit(20)
)

company,job_title,offices,office
Cloudflare,"Account Executive, FedCiv","List(List(List(), 19994, Washington, DC, United States, Washington, DC, 48752))","Washington, DC"
Cloudflare,"Account Executive, Indonesia","List(List(List(), 19992, Singapore, Singapore, 74844))",Singapore
Cloudflare,AI Security Research & Red Team Engineer,"List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752), List(List(), 19990, New York, New York, United States, New York, NY, 48752))","Austin, TX"
Cloudflare,"Associate General Counsel, Privacy Compliance","List(List(List(), 19988, London, United Kingdom, London, United Kingdom, 74843))","London, United Kingdom"
Cloudflare,Business Development Representative - Bahasa Indonesia Speaking,"List(List(List(), 19992, Singapore, Singapore, 74844))",Singapore
Cloudflare,Business Development Representative (BDR),"List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))","Austin, TX"
Cloudflare,"Business Development Representative (BDR), LATAM Portuguese Speaking","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))","Austin, TX"
Cloudflare,"Business Development Representative (BDR), LATAM Spanish Speaking","List(List(List(), 19983, Austin, TX, United States, Austin, TX, 48752))","Austin, TX"
Cloudflare,"Business Development Representative, Bengaluru","List(List(List(), 83149, null, Remote India, 74844))",Remote India
Cloudflare,Business Development Representative (Cantonese Speaking),"List(List(List(), 48755, Beijing Shi, China, Beijing, China, 74844), List(List(), 19992, Singapore, Singapore, 74844))","Beijing, China"


In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "employment_type",
        F.when(
            F.col("employment").isNotNull(),
            F.trim(
                F.regexp_replace(
                    F.col("employment"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "posted_at",
        F.to_timestamp("first_published")
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    )
)

In [0]:
greenhouse_silver.select(
    "job_id",
    "job_title",
    "posted_at",
    "updated_at"
).show(10, truncate=False)

+-------+--------------------------------------------------------------------+-------------------+-------------------+
|job_id |job_title                                                           |posted_at          |updated_at         |
+-------+--------------------------------------------------------------------+-------------------+-------------------+
|7695702|Account Executive, FedCiv                                           |2026-03-09 20:05:19|2026-10-02 18:24:13|
|8172582|Account Executive, Indonesia                                        |2026-09-08 01:25:04|2026-10-02 18:24:13|
|8097321|AI Security Research & Red Team Engineer                            |2026-08-10 15:02:09|2026-10-02 18:24:13|
|8144669|Associate General Counsel, Privacy Compliance                       |2026-08-21 17:54:32|2026-10-02 18:24:13|
|7167659|Business Development Representative - Bahasa Indonesia Speaking     |2025-08-18 01:32:53|2026-10-02 18:24:13|
|6557318|Business Development Representative (BD

In [0]:
greenhouse_silver.select(
    "job_id",
    "job_title",
    "posted_at",
    "updated_at"
).show(10, truncate=False)

+-------+--------------------------------------------------------------------+-------------------+-------------------+
|job_id |job_title                                                           |posted_at          |updated_at         |
+-------+--------------------------------------------------------------------+-------------------+-------------------+
|7695702|Account Executive, FedCiv                                           |2026-03-09 20:05:19|2026-10-02 18:24:13|
|8172582|Account Executive, Indonesia                                        |2026-09-08 01:25:04|2026-10-02 18:24:13|
|8097321|AI Security Research & Red Team Engineer                            |2026-08-10 15:02:09|2026-10-02 18:24:13|
|8144669|Associate General Counsel, Privacy Compliance                       |2026-08-21 17:54:32|2026-10-02 18:24:13|
|7167659|Business Development Representative - Bahasa Indonesia Speaking     |2025-08-18 01:32:53|2026-10-02 18:24:13|
|6557318|Business Development Representative (BD

In [0]:
greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "is_remote",
        F.when(
            F.lower(F.coalesce(F.col("location"), F.lit("")))
            .rlike(r"\bremote\b"),
            F.lit(True)
        )
        .otherwise(F.lit(False))
    )
)

In [0]:
greenhouse_silver.select(
    F.sum(
        F.col("job_id").isNull().cast("int")
    ).alias("null_job_id"),

    F.sum(
        F.col("job_title").isNull().cast("int")
    ).alias("null_job_title"),

    F.sum(
        F.col("company").isNull().cast("int")
    ).alias("null_company")
).show()

+-----------+--------------+------------+
|null_job_id|null_job_title|null_company|
+-----------+--------------+------------+
|          0|             0|           0|
+-----------+--------------+------------+



In [0]:
greenhouse_silver.select(
    F.sum(
        F.col("job_id").isNull().cast("int")
    ).alias("null_job_id"),

    F.sum(
        F.col("job_title").isNull().cast("int")
    ).alias("null_job_title"),

    F.sum(
        F.col("company").isNull().cast("int")
    ).alias("null_company")
).show()

+-----------+--------------+------------+
|null_job_id|null_job_title|null_company|
+-----------+--------------+------------+
|          0|             0|           0|
+-----------+--------------+------------+



In [0]:
window_spec = Window.partitionBy("job_id").orderBy(
    F.col("updated_at").desc_nulls_last(),
    F.col("ingestion_timestamp").desc()
)

greenhouse_silver = (
    greenhouse_silver
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
greenhouse_silver_final = greenhouse_silver.select(
    # Lineage
    "source",
    "company",
    "board_token",
    "ingestion_timestamp",
    "ingestion_date",

    # Job identity
    "job_id",
    "job_title",

    # Job classification
    "department",
    "office",
    "employment_type",

    # Location
    "location",
    "location_normalized",
    "is_remote",

    # Description
    "content",

    # Dates
    "posted_at",
    "updated_at",

    # URL
    "job_url"
)

In [0]:
greenhouse_silver_final.printSchema()

root
 |-- source: string (nullable = true)
 |-- company: string (nullable = true)
 |-- board_token: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- job_id: long (nullable = true)
 |-- job_title: string (nullable = true)
 |-- department: string (nullable = true)
 |-- office: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- location_normalized: string (nullable = true)
 |-- is_remote: boolean (nullable = false)
 |-- content: string (nullable = true)
 |-- posted_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- job_url: string (nullable = true)



In [0]:
print(
    "Greenhouse Silver records:",
    greenhouse_silver_final.count()
)

Greenhouse Silver records: 1583


In [0]:
greenhouse_silver_final.groupBy("job_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+------+-----+
|job_id|count|
+------+-----+
+------+-----+



In [0]:
greenhouse_silver_final.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in greenhouse_silver_final.columns
]).show()

+------+-------+-----------+-------------------+--------------+------+---------+----------+------+---------------+--------+-------------------+---------+-------+---------+----------+-------+
|source|company|board_token|ingestion_timestamp|ingestion_date|job_id|job_title|department|office|employment_type|location|location_normalized|is_remote|content|posted_at|updated_at|job_url|
+------+-------+-----------+-------------------+--------------+------+---------+----------+------+---------------+--------+-------------------+---------+-------+---------+----------+-------+
|     0|      0|          0|                  0|             0|     0|        0|         0|     0|           1340|       0|                  0|        0|      0|        0|         0|      0|
+------+-------+-----------+-------------------+--------------+------+---------+----------+------+---------------+--------+-------------------+---------+-------+---------+----------+-------+



In [0]:
(
    greenhouse_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "job_market.silver.greenhouse_jobs"
    )
)

In [0]:
spark.sql("""
    SELECT
        company,
        COUNT(*) AS job_count
    FROM job_market.silver.greenhouse_jobs
    GROUP BY company
    ORDER BY job_count DESC
""").display()

company,job_count
Stripe,719
Cloudflare,416
Coinbase,226
Figma,161
GuidePoint Security,61
